# PHASE 1 : WEB SCRAPING (Python)

In [3]:
# Importing the prerequisites
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from urllib.parse import urljoin

In [4]:
# URL Link
url = "https://books.toscrape.com/"

In [5]:
# Step-1: Fetch all genres
response = requests.get(url) # Snding HTTP request
response.encoding = "utf-8" # UTF encoding
soup = BeautifulSoup(response.text, "html.parser") # Parsing HTML using BeautifulSoup

In [6]:
# Fetching genre links
categories = soup.select("div.side_categories ul li ul li a")

In [7]:
# Creating a genre dictionary to store genre name and genre URL
genre_links = {}
for cat in categories:
    genre = cat.text.strip()
    link = urljoin(url, cat["href"])
    genre_links[genre] = link

print(f"✅ Total genres found: {len(genre_links)}")

✅ Total genres found: 50


In [8]:
# Step-2: Scrape all books genre-wise
data = []

rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

In [11]:
for genre, genre_url in genre_links.items():
    print(f"\n📖 Scraping genre: {genre}")
    page_url = genre_url

    while True:
        response = requests.get(page_url) # sending request to genre url
        response.encoding = "utf-8" # UTF encoding
        soup = BeautifulSoup(response.text, "html.parser") # Parsing HTML using BeautifulSoup

        books = soup.select("article.product_pod")

        if not books: # exit loop if no books are found on the page
            break

        print("Books found on page:", len(books)) # print number of books found on the current page

        for book in books:
            # Title
            title = book.h3.a["title"]

            # Price
            price_text = book.select_one(".price_color").get_text()
            price = price_text.replace("£", "").replace("Â", "").strip()

            # Rating
            rating_word = book.p["class"][1]
            rating = rating_map.get(rating_word)

            # Availability (categorical)
            availability = book.select_one(".availability").get_text(strip=True)

            data.append([
                title,
                genre,
                float(price),
                rating,
                availability
            ])

        # Pagination
        next_btn = soup.select_one("li.next a")
        if next_btn:
            page_url = urljoin(page_url, next_btn["href"])
            time.sleep(0.5)
        else:
            break


📖 Scraping genre: Travel
Books found on page: 11

📖 Scraping genre: Mystery
Books found on page: 20
Books found on page: 12

📖 Scraping genre: Historical Fiction
Books found on page: 20
Books found on page: 6

📖 Scraping genre: Sequential Art
Books found on page: 20
Books found on page: 20
Books found on page: 20
Books found on page: 15

📖 Scraping genre: Classics
Books found on page: 19

📖 Scraping genre: Philosophy
Books found on page: 11

📖 Scraping genre: Romance
Books found on page: 20
Books found on page: 15

📖 Scraping genre: Womens Fiction
Books found on page: 17

📖 Scraping genre: Fiction
Books found on page: 20
Books found on page: 20
Books found on page: 20
Books found on page: 5

📖 Scraping genre: Childrens
Books found on page: 20
Books found on page: 9

📖 Scraping genre: Religion
Books found on page: 7

📖 Scraping genre: Nonfiction
Books found on page: 20
Books found on page: 20
Books found on page: 20
Books found on page: 20
Books found on page: 20
Books found on page: 1

In [12]:
# Save to CSV
df = pd.DataFrame(
    data,
    columns=[
        "Title",
        "Genre",
        "Price",
        "Rating",
        "Availability"
    ]
)

In [13]:
df.to_csv("books_full_dataset.csv", index=False)

print("\n✅ Full Scrapping Completed")
print("📊 Total books scraped:", len(df))
print(df.head())


✅ Full Scrapping Completed
📊 Total books scraped: 1000
                                               Title   Genre  Price  Rating  \
0                            It's Only the Himalayas  Travel  45.17       2   
1  Full Moon over Noah’s Ark: An Odyssey to Mount...  Travel  49.43       4   
2  See America: A Celebration of Our National Par...  Travel  48.87       3   
3  Vagabonding: An Uncommon Guide to the Art of L...  Travel  36.94       2   
4                               Under the Tuscan Sun  Travel  37.33       3   

  Availability  
0     In stock  
1     In stock  
2     In stock  
3     In stock  
4     In stock  
